# Part 0 — LangChain Warmup
**Context:** You are a Data Business Analyst at a global beverage FMCG company.  
Each cell introduces one core concept, then applies it to a Supply Chain / O2C scenario.

**Prerequisites:** Run this first:
```bash
micromamba create -f environment.yml
micromamba activate langchain-workshop
```
Then copy `.env.example` → `.env` and add your `OPENAI_API_KEY`.

---
## Cell 1 — Load API key from .env
**Concept:** `python-dotenv` reads `KEY=value` pairs from `.env` into `os.environ`.  
This keeps secrets out of notebooks and git history.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in current directory

# Verify the key loaded (prints only first 8 chars for safety)
key = os.getenv('OPENAI_API_KEY', 'NOT SET')
print(f"Key loaded: {key[:8]}..." if key != 'NOT SET' else "ERROR: OPENAI_API_KEY not set")

MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
print(f"Using model: {MODEL}")

---
## Cell 2 — ChatModel: the LLM wrapper
**Concept:** `ChatOpenAI` is LangChain's wrapper around OpenAI's chat endpoint.  
Key parameters:
- `model` — which model to call
- `temperature` — 0 = deterministic, 1 = creative. **Use 0 for data extraction, ~0.3 for drafting.**

**SC3 relevance:** When extracting structured data from supplier emails or PO documents,  
always set `temperature=0` — you want the same answer every time for auditability.

In [ ]:
from langchain_openai import ChatOpenAI

# temperature=0: deterministic — right for data extraction tasks
llm = ChatOpenAI(model=MODEL, temperature=0)

# Direct call — simplest possible usage
response = llm.invoke("In one sentence: what is Order-to-Cash (O2C)?")
print(response.content)

---
## Cell 3 — PromptTemplate: parameterised prompts
**Concept:** A `ChatPromptTemplate` separates *prompt structure* from *runtime values*.  
The `{placeholder}` syntax injects variables at call time.

**Why it matters for a BA:**  
This is the LangChain equivalent of writing a functional requirement with parameters.  
The template IS the specification — version-control it like code.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Template for classifying supply chain anomalies
anomaly_template = ChatPromptTemplate.from_template("""
You are a supply chain analyst at a global beverage manufacturer.
Classify the following demand forecast anomaly and suggest the most likely root cause.

SKU: {sku}
Market: {market}
Anomaly: {anomaly_description}

Respond in exactly this format:
Classification: <one of: Demand Spike, Demand Drop, Data Error, Seasonal Shift>
Likely cause: <one sentence>
Recommended action: <one sentence>
""")

# Format the prompt — this just builds the message, no LLM call yet
messages = anomaly_template.format_messages(
    sku="TEA-GB-EarlGrey-100",
    market="United Kingdom",
    anomaly_description="Forecast shows +340% spike in week 48 vs 4-week average"
)

print("--- Prompt sent to model ---")
print(messages[0].content)

---
## Cell 4 — LCEL: the pipe operator `|`
**Concept:** LangChain Expression Language (LCEL) chains steps with `|`.  
`prompt | model | parser` is a **Runnable** — it has `.invoke()`, `.batch()`, `.stream()`.

**Interview talking point:**  
> "LCEL is LangChain's composition layer — like a dbt DAG but for LLM calls.  
> Each step is independently testable, which matters for UAT: I can mock the model  
> and test just the prompt template, or test just the parser against known outputs."

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Build the chain: prompt → model → string parser
anomaly_chain = anomaly_template | llm | StrOutputParser()

# Invoke with a single input
result = anomaly_chain.invoke({
    "sku": "BEV-DE-Sparkling-500ml",
    "market": "Germany",
    "anomaly_description": "Forecast shows -60% drop starting week 2 of January"
})

print(result)

---
## Cell 5 — batch(): processing multiple items
**Concept:** `.batch()` runs the chain on a list of inputs **in parallel** (configurable concurrency).  
Critical for production SC use cases — processing 500 SKUs, not 1.

**NFR talking point for interviews:**  
> "A non-functional requirement for any demand anomaly detection feature is throughput:  
> the system must classify all anomalies for a weekly S&OP run within 5 minutes.  
> `.batch()` with `max_concurrency=10` is how LangChain meets that NFR."

In [ ]:
# Simulate a batch of anomalies from the weekly S&OP data feed
weekly_anomalies = [
    {
        "sku": "TEA-GB-EarlGrey-100",
        "market": "United Kingdom",
        "anomaly_description": "+340% spike in week 48 — Black Friday promotional period"
    },
    {
        "sku": "BEV-FR-StillWater-1L",
        "market": "France",
        "anomaly_description": "Negative forecast value (-200 cases) in week 3"
    },
    {
        "sku": "JUC-AU-OrangeJuice-2L",
        "market": "Australia",
        "anomaly_description": "-45% drop aligned with competitor launching lower-price SKU"
    },
]

# Process all three in parallel — note max_concurrency to avoid rate limits
results = anomaly_chain.batch(weekly_anomalies, config={"max_concurrency": 3})

for anomaly, result in zip(weekly_anomalies, results):
    print(f"\n=== {anomaly['sku']} ({anomaly['market']}) ===")
    print(result)

---
## Cell 6 — StructuredOutputParser: typed extraction
**Concept:** Instead of a raw string, force the model to return a Python dict  
with a defined schema. LangChain injects format instructions into the prompt.

**BA relevance:**  
This is your acceptance criterion mechanism — the output parser IS the DoD for  
the extraction feature. If the parser fails, the test fails.

In [ ]:
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

# Define the schema — this becomes part of the prompt automatically
response_schemas = [
    ResponseSchema(name="classification",
                   description="One of: Demand Spike, Demand Drop, Data Error, Seasonal Shift",
                   type="string"),
    ResponseSchema(name="root_cause",
                   description="Most likely business reason for the anomaly",
                   type="string"),
    ResponseSchema(name="action",
                   description="Recommended action for the supply planner",
                   type="string"),
    ResponseSchema(name="confidence",
                   description="Confidence score 0-10 for this classification",
                   type="integer"),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

# Show what gets injected into the prompt
print("Format instructions injected into prompt:")
print(format_instructions)

In [ ]:
# Build a structured extraction chain
structured_template = ChatPromptTemplate.from_template("""
You are a supply chain analyst at a global beverage manufacturer.
Classify this demand forecast anomaly.

SKU: {sku}
Market: {market}
Anomaly: {anomaly_description}

{format_instructions}
""")

structured_chain = structured_template | llm | output_parser

result = structured_chain.invoke({
    "sku": "TEA-GB-EarlGrey-100",
    "market": "United Kingdom",
    "anomaly_description": "+340% spike in week 48 — Black Friday promotional period",
    "format_instructions": format_instructions
})

print(type(result))  # dict — not a string!
print(result)
print(f"\nConfidence: {result['confidence']}/10")
print(f"Action: {result['action']}")

---
## Cell 7 — Multi-step chain: chaining two LLM calls
**Concept:** Output of chain 1 becomes input of chain 2.  
Pattern: `chain1 | {key: RunnablePassthrough()} | chain2`

**SC3 scenario:**  
Step 1 — classify the anomaly (fast, cheap model).  
Step 2 — draft the Azure DevOps work item description from the classification.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Chain 1: classify anomaly → plain string
classify_template = ChatPromptTemplate.from_template("""
Classify this supply chain anomaly in one sentence.
SKU: {sku}, Market: {market}
Anomaly: {anomaly_description}
""")
classify_chain = classify_template | llm | StrOutputParser()

# Chain 2: turn classification into an Azure DevOps user story
azdo_template = ChatPromptTemplate.from_template("""
You are a Business Analyst writing Azure DevOps work items.
Based on this supply chain anomaly classification:

{classification}

Write a User Story in INVEST format:
- Title: As a [role], I want [goal] so that [benefit]
- Acceptance Criteria (3 bullet points)
- Definition of Done (2 bullet points)
""")
azdo_chain = azdo_template | llm | StrOutputParser()

# Full pipeline: input → classify → draft AzDO story
full_pipeline = classify_chain | (lambda classification: {"classification": classification}) | azdo_chain

story = full_pipeline.invoke({
    "sku": "TEA-GB-EarlGrey-100",
    "market": "United Kingdom",
    "anomaly_description": "+340% spike in week 48 — Black Friday promotional period"
})

print(story)

---
## Cell 8 — streaming: token-by-token output
**Concept:** `.stream()` yields tokens as they arrive — important for UX on long responses.  
**NFR:** For a planning assistant UI, streaming is the difference between  
a 10-second blank screen and visible progress. Document this as an NFR in the BRD.

In [ ]:
stream_template = ChatPromptTemplate.from_template("""
Write a 3-paragraph executive summary of the supply chain risks 
for a global beverage manufacturer entering the {market} market 
with a new {product_type} product line.
""")

stream_chain = stream_template | llm | StrOutputParser()

print("Streaming response:")
print("-" * 40)
for chunk in stream_chain.stream({"market": "India", "product_type": "premium iced tea"}):
    print(chunk, end="", flush=True)
print("\n" + "-" * 40)

---
## Cell 9 — Temperature experiment
**Concept:** Same prompt, different temperatures → different outputs.  
Run this cell twice with different temps and observe the difference.

| Temperature | Use case |
|---|---|
| 0.0 | Data extraction, classification, SQL generation |
| 0.3 | Drafting work items, summaries |
| 0.7 | Creative content, brainstorming |
| 1.0+ | Very creative — rarely useful in enterprise SC |


In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Suggest a catchy product name for a new premium tea blend targeting health-conscious consumers in {market}."
)

for temp in [0.0, 0.7, 1.2]:
    creative_llm = ChatOpenAI(model=MODEL, temperature=temp)
    chain = prompt | creative_llm | StrOutputParser()
    result = chain.invoke({"market": "United Kingdom"})
    print(f"Temperature {temp}: {result}")

---
## Cell 10 — Summary: concept map to SC3 role

| LangChain concept | What you built | JD requirement |
|---|---|---|
| `ChatOpenAI(temperature=0)` | Deterministic extractor | Data quality, auditability |
| `ChatPromptTemplate` | Parameterised spec | Functional requirements |
| `prompt \| model \| parser` (LCEL) | Composable pipeline | Production-ready AI solutions |
| `.batch(max_concurrency=N)` | Bulk SKU processing | NFR: throughput for S&OP |
| `StructuredOutputParser` | Typed extraction schema | DoD / acceptance criteria |
| Multi-step chain | Classify → draft AzDO story | Agentic workflow |
| `.stream()` | Progressive UI output | NFR: user experience |
| Temperature comparison | Determinism vs creativity tradeoff | BA judgement call |

**Next:** Open `Part3_Agents` to learn Tools & Agents — the highest-leverage topic for this role.